# Churn Motive Scoring (21.0)

Runs after `10.0_batch_score_churn`: for **every client with a churn score** (regardless of prediction), the four registered motive models (price / competitor / payment / technical, `workspace.default.motive_*@champion`) estimate the likelihood of each churn motive — consumers downstream decide what to do with that information. Results append to `workspace.default.churn_motives` with provenance (`source_file`, `scored_at`) and a null-safe feature digest, so a client is re-scored only when their feature line actually changes.

In [0]:
%pip install lightgbm

In [0]:
# --- Parameters (widget defaults work for both manual runs and the scheduled job) ---
dbutils.widgets.text("scores_table", "workspace.default.churn_scores")
dbutils.widgets.text("motives_table", "workspace.default.churn_motives")
dbutils.widgets.text("catalog_name", "workspace.default")

SCORES_TABLE = dbutils.widgets.get("scores_table")
MOTIVES_TABLE = dbutils.widgets.get("motives_table")
CATALOG_NAME = dbutils.widgets.get("catalog_name")

print(f"Scores table:  {SCORES_TABLE}")
print(f"Motives table: {MOTIVES_TABLE}")
print(f"Model catalog: {CATALOG_NAME}")

In [0]:
import mlflow.lightgbm
import pandas as pd
from pyspark.sql import functions as F, Window

# The 37 motive-model features (training X from 20.1_shap_motives, sentiment probs excluded)
MOTIVE_FEATURES = [
    # Demographics
    "isMale", "SeniorCitizen", "isPartner", "HasDependents", "MonthlyIncome",
    # Account & tenure
    "nbrTenure", "isNewCustomer", "isPaperlessBilling", "isContractOneYear", "isContractTwoYear",
    # Payment methods
    "isPaymentMethod_ElectronicCheck", "isPaymentMethod_MailedCheck",
    "isPaymentMethod_BankTransfer", "isPaymentMethod_CreditCard",
    # Core telecom services
    "hasPhoneService", "hasMultipleLines", "hasInternetService",
    "isInternetService_DSL", "isInternetService_Fiber",
    # Add-on services
    "hasOnlineSecurity", "hasOnlineBackup", "hasDeviceProtection", "hasTechSupport",
    "hasStreamingTV", "hasStreamingMovies", "nbrInternetServicesAddOnQty", "hasSecurityBundle",
    # Business interactions
    "isM2M_dual", "isM2M_internetOnly", "isOneYear_dual", "isTwoYear_dual",
    # Billing & financial ratios
    "MonthlyCharges", "TotalCharges", "TotalSpend", "chargeRatio", "numServices", "chargesPerService",
]
MOTIVES = ["preco", "concorrente", "pagamento", "tecnico"]

# Load the @champion version of each motive model from Unity Catalog.
# Direct sklearn predict (not pyfunc) so NaN features are handled natively by LightGBM
# instead of being rejected by signature enforcement.
motive_models = {
    m: mlflow.lightgbm.load_model(f"models:/{CATALOG_NAME}.motive_{m}@champion") for m in MOTIVES
}

# Verify each model expects exactly these features
for m, model in motive_models.items():
    fn = list(getattr(model, "feature_names_in_", model.feature_name_))
    assert fn == MOTIVE_FEATURES, f"motive_{m} expects different features: {set(fn) ^ set(MOTIVE_FEATURES)}"
print(f"Loaded {len(motive_models)} champion motive models, all aligned on {len(MOTIVE_FEATURES)} features.")

In [0]:
# --- Who to score: latest churn score per client — EVERYONE, not just predicted churners ---
# Motive probabilities are computed for every scored client; downstream consumers decide how
# to use them (the models were trained on churners, so they are most informative for
# at-risk clients, but nothing is hidden from consumers).
# Dedup follows the same whole-line philosophy as 10.0: a null-safe md5 digest over the 37
# motive features identifies the client's feature line. A client is re-scored only when that
# line changes; identical repeats are skipped.
def feature_digest(df):
    return df.withColumn("feature_digest", F.md5(F.concat_ws("\x1f", *[
        F.coalesce(F.col(c).cast("string"), F.lit("\x00NULL")) for c in MOTIVE_FEATURES
    ])))

scores = spark.table(SCORES_TABLE)
latest = (feature_digest(scores)
    .withColumn("_rn", F.row_number().over(
        Window.partitionBy("customerID").orderBy(F.col("scored_at").desc())))
    .filter("_rn = 1")
    .drop("_rn"))
to_score = latest  # everyone — no prediction filter; consumers decide what to use

if spark.catalog.tableExists(MOTIVES_TABLE):
    already = spark.table(MOTIVES_TABLE).select("customerID", "feature_digest")
    to_score = to_score.join(already, ["customerID", "feature_digest"], "left_anti")

n_to_score = to_score.count()
print(f"Clients to motive-score this round: {n_to_score}")
if n_to_score == 0:
    dbutils.notebook.exit("No new or changed clients — nothing to score.")

In [0]:
# --- Predict the 4 motive probabilities with the champion models and append to the output table ---
pdf = to_score.select("customerID", "source_file", "feature_digest", *MOTIVE_FEATURES).toPandas()
X = pdf[MOTIVE_FEATURES]  # NaNs are fine — LightGBM handles missing values natively

for m in MOTIVES:
    pdf[f"prob_{m}"] = motive_models[m].predict(X)

# Dominant motive = highest predicted probability
prob_cols = [f"prob_{m}" for m in MOTIVES]
pdf["top_motive"] = pdf[prob_cols].idxmax(axis=1).str.replace("prob_", "", regex=False)

result = (spark.createDataFrame(pdf)
    .select("customerID", *prob_cols, "top_motive", "feature_digest", "source_file")
    .withColumn("scored_at", F.current_timestamp()))

(result.write.format("delta").mode("append").saveAsTable(MOTIVES_TABLE))
print(f"Appended {result.count()} motive scores to {MOTIVES_TABLE}")
display(result.limit(5))

In [0]:
# --- Sanity checks ---
motives_tbl = spark.table(MOTIVES_TABLE)
print(f"Total rows in {MOTIVES_TABLE}: {motives_tbl.count()}")

print("\nTop-motive distribution (all time):")
motives_tbl.groupBy("top_motive").count().orderBy(F.desc("count")).show()

# Every client's current feature line must be motive-scored (recompute the gate)
latest_chk = (feature_digest(spark.table(SCORES_TABLE))
    .withColumn("_rn", F.row_number().over(
        Window.partitionBy("customerID").orderBy(F.col("scored_at").desc())))
    .filter("_rn = 1").drop("_rn"))
remaining = (latest_chk
    .join(motives_tbl.select("customerID", "feature_digest"), ["customerID", "feature_digest"], "left_anti"))
print(f"Client feature lines still missing motive scores: {remaining.count()}")